# HW3 練功本：把 `st_minecode.ipynb` 拆成小積木來懂

這本是**給我自己看的**，不用交。它和正式的 `st_minecode.ipynb` 分開，原因是：

1. 正式 notebook 要讀 1 GB 的維基、下載 3 個模型、跑 GPU 40 分鐘，不適合「改一行、跑一下」來理解。
2. 這本只用**幾行假資料**，CPU 幾秒就跑完（本機、Colab 免 GPU 都可以），可以隨便改、隨便印。
3. 每一節都是同一個格式：**白話解說 → 🧪 示範（直接跑）→ ✍️ 手打練習（把 `...` 換成自己的程式）→ 參考答案（點開才看得到）**。

練習格子跑的時候會自動對答案：⏳ = 還沒寫、✅ = 對了、❌ = 不對（會印出期望的結果）。

| 節 | 主題 | 對應正式 notebook 的位置 | 建議 |
| :--- | :--- | :--- | :--- |
| 1 | 證據格式、`gold_groups`、空集合陷阱 | 第 2 節開頭的工具函式 | 看懂就好 |
| 2 | 標題比對 `match_titles` | 第 2 節「標題比對」 | ✍️ **手打** |
| 3 | pooling、L2 normalize、cosine | 第 2 節 `encode_texts` | ✍️ **手打** |
| 4 | 矩陣乘法、`topk`、混合分數 | 第 2 節 `page_candidates` / `rank_pages` | ✍️ **手打** |
| 5 | local index → 全域 index | 第 3 節 `rank_sentences_dense` | ✍️ **手打** |
| 6 | bi-encoder vs cross-encoder | 第 3 節 reranker | 看懂就好 |
| 7 | 訓練資料：補進標準證據 | 第 3 節最後「準備分類器的輸入文字」 | ✍️ **手打** |
| 8 | 訓練 5 步驟 + fp16 + warmup | 第 4 節 Training | ✍️ **手打** |
| 9 | Recall、FEVER score、NEI bias | 第 2～5 節的評估 | ✍️ **手打** |
| 10 | submission 格式 | 第 6 節 | 看懂就好 |

In [ ]:
import json
import re

import torch
import torch.nn.functional as F

torch.manual_seed(0)


def check(name, got, expected):
    """練習格子用的對答案小工具。"""
    if got is None or got is Ellipsis:
        print(f'⏳ {name}：還沒寫完（把 ... 換成你的程式）')
        return
    same = torch.allclose(got, expected, atol=1e-5) if isinstance(expected, torch.Tensor) else got == expected
    print(f'✅ {name}：正確！' if same else f'❌ {name}：不對\n   你的結果：{got}\n   期望結果：{expected}')

## 1. 標準證據長什麼樣子？`gold_groups` 在做什麼？

一筆 claim 的 `evidence` 有三層中括號：

```text
evidence = [                                   ← 第 1 層：好幾「組」可以接受的答案（找到任一組就算對）
    [ [101, 1, '周杰倫', 0] ],                  ← 第 2 層：一組（這組只要 1 句）
    [ [101, 2, '周杰倫', 3], [101, 3, '葉惠美', 1] ],   ← 這組要 2 句「一起」找到
]                    ↑第 3 層：一條證據 = [標註 id, 證據 id, 頁名, 句號]
```

`gold_groups` 把它整理成「一組一個 set」：`[{('周杰倫', 0)}, {('周杰倫', 3), ('葉惠美', 1)}]`，之後就能用 `<=`（issubset）判斷「這組是不是全部被找到了」。

NEI 的證據長 `[[[101, None, None, None]]]`，頁名是 `None`，要跳過。**還有一個陷阱：空集合是任何集合的子集合**，
如果讓空組混進來，`set() <= 任何東西` 永遠是 `True`，沒找到證據也會被算成命中。

In [ ]:
# 🧪 示範
evidence = [[[101, 1, '周杰倫', 0]], [[101, 2, '周杰倫', 3], [101, 3, '葉惠美', 1]]]
groups = [{(item[2], item[3]) for item in group} for group in evidence]
print('整理後：', groups)

found = {('周杰倫', 3), ('葉惠美', 1), ('李安', 0)}
print('第 1 組有沒有找齊？', groups[0] <= found)   # ('周杰倫', 0) 不在 found 裡 → False
print('第 2 組有沒有找齊？', groups[1] <= found)   # 兩句都在 → True
print('任一組找齊就算對：', any(group <= found for group in groups))
print('⚠️ 空集合陷阱：set() <= {"A"} =', set() <= {'A'}, '，set() <= set() =', set() <= set())

## 2. 標題比對：claim 的每個子字串都去查表 ✍️

想法很單純：維基頁名常常就是 claim 的主角。把所有頁名做成 dict（查詢是 O(1)，就像 C# 的 `Dictionary<string, List<string>>`），
再把 claim 的每個子字串拿去查。claim 長 n 個字，起點有 n 種、長度最多 30 種，所以一題最多查 30n 次，一百萬個標題也很快。

兩個細節：
- `title_key` 用正規表示式去掉結尾的消歧義括號：`r'\s*[（(][^（）()]*[）)]\s*$'` = 「可能有空白 → 左括號（全形或半形）→ 一串不是括號的字 → 右括號 → 字串結尾」。
- 同一個頁面可能被不同長度的子字串命中，記最長的那個；最後**長標題排前面**（「周杰倫」比「周杰」具體）。

In [ ]:
# 🧪 示範：title_key 的正規表示式
for page_id in ['水星 (行星)', '水星（神話）', '周杰倫', 'iPhone 6']:
    title = re.sub(r'\s*[（(][^（）()]*[）)]\s*$', '', page_id)   # 去掉結尾括號
    key = re.sub(r'\s+', '', title.lower())                        # 小寫、去空白
    print(f'{page_id!r:16} → {key!r}')

# 🧪 示範：一個迷你的「標題 → 頁面」索引
title_index = {'周杰倫': ['周杰倫'], '周杰': ['周杰'], '葉惠美': ['葉惠美'], '專輯': ['專輯'],
               '水星': ['水星 (行星)', '水星 (神話)']}
claim = '周杰倫的專輯葉惠美提到水星'
for start in range(len(claim)):
    for end in range(start + 2, min(len(claim), start + 30) + 1):
        if claim[start:end] in title_index:
            print(f'  claim[{start}:{end}] = {claim[start:end]!r} → {title_index[claim[start:end]]}')

In [ ]:
# ✍️ 手打練習：寫出 match_titles(claim)
# 規則：子字串長度 2～30；found 記錄「頁名 → 命中的最長長度」；最後依長度由長到短排序（同長度照頁名排序）。
def match_titles(claim):
    found = {}
    ...  # 雙層 for：start、end；用 title_index.get(子字串, []) 拿頁面清單
    return None  # 寫完後改成：sorted(found, key=lambda page_id: (-found[page_id], page_id))


check('match_titles', match_titles(claim), ['周杰倫', '葉惠美', '周杰', '專輯', '水星 (神話)', '水星 (行星)'])

<details><summary>參考答案（先自己寫再點開）</summary>

```python
def match_titles(claim):
    found = {}
    for start in range(len(claim)):
        for end in range(start + 2, min(len(claim), start + 30) + 1):
            for page_id in title_index.get(claim[start:end], []):
                found[page_id] = max(found.get(page_id, 0), end - start)
    return sorted(found, key=lambda page_id: (-found[page_id], page_id))
```

為什麼用 `title_index.get(key, [])` 而不是 `title_index[key]`？查不到時 `[key]` 會丟 `KeyError`，`.get` 會回傳預設值 `[]`，for 迴圈就直接跳過。
</details>

## 3. 文字 → 向量：pooling、L2 normalize、cosine ✍️

BERT 類模型會輸出 `last_hidden_state`，形狀 `[batch, token 數, 維度]`：**每個 token 一個向量**。要變成「一段文字一個向量」有兩種常見做法：

| 做法 | 程式 | 誰在用 |
| :--- | :--- | :--- |
| CLS pooling | `hidden[:, 0]`（取第 0 個 token） | bge-m3（它訓練時就是這樣用的） |
| mean pooling | 只平均 `attention_mask = 1` 的 token | sample code 的 bert-base-chinese |

mean pooling 要小心 padding：同一批文字長度不同，短的會補 `[PAD]`，這些位置不能算進平均。
`mask.unsqueeze(-1)` 把 `[batch, token]` 變成 `[batch, token, 1]`，乘上 `[batch, token, 維度]` 時會自動複製到每一維（broadcasting）。

最後 `F.normalize(..., p=2, dim=1)` 把每個向量長度變成 1。cosine = 內積 ÷ (長度 × 長度)，長度都是 1，所以**內積就等於 cosine**，一個矩陣乘法就能算出所有相似度。

In [ ]:
# 🧪 示範：2 段文字，第 1 段有 3 個真 token，第 2 段只有 2 個真 token + 1 個 padding；每個 token 是 2 維向量
hidden = torch.tensor([[[1., 1.], [3., 1.], [2., 4.]],
                       [[2., 0.], [4., 2.], [9., 9.]]])   # 第 2 段最後的 [9, 9] 是 padding，不該被算進去
mask = torch.tensor([[1, 1, 1],
                     [1, 1, 0]])
print('hidden 形狀：', tuple(hidden.shape), '| mask 形狀：', tuple(mask.shape), '→ unsqueeze 後', tuple(mask.unsqueeze(-1).shape))
print('CLS pooling：', hidden[:, 0].tolist())
print('錯誤示範（連 padding 一起平均）：', hidden.mean(dim=1).tolist())

In [ ]:
# ✍️ 手打練習：mean pooling（排除 padding）+ L2 normalize
def mean_pool_and_normalize(hidden, mask):
    mask = ...      # 1. mask 增加最後一維並轉成 float
    pooled = ...    # 2. (hidden * mask) 沿 token 維度加總，再除以真 token 數（用 clamp(min=1) 防止除以 0）
    return None     # 3. 改成 F.normalize(pooled, p=2, dim=1)


expected = F.normalize(torch.tensor([[2., 2.], [3., 1.]]), p=2, dim=1)
check('mean pooling', mean_pool_and_normalize(hidden, mask), expected)

<details><summary>參考答案</summary>

```python
def mean_pool_and_normalize(hidden, mask):
    mask = mask.unsqueeze(-1).float()                                # [2, 3] → [2, 3, 1]
    pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
    return F.normalize(pooled, p=2, dim=1)
```
第 1 段：(1+3+2)/3 = 2、(1+1+4)/3 = 2 → [2, 2]；第 2 段：(2+4)/2 = 3、(0+2)/2 = 1 → [3, 1]（padding 的 [9, 9] 被 mask 乘成 0）。
</details>

## 4. 矩陣乘法、`topk`、混合分數 ✍️

claim 向量 `[claim 數, 維度]` @ 頁面向量轉置 `[維度, 頁數]` = `[claim 數, 頁數]`：**第 i 列第 j 欄 = 第 i 個 claim 和第 j 頁的 cosine**。
`topk(k, dim=1)` 沿著每一列挑最大的 k 個，回傳 `values`（分數）和 `indices`（第幾頁）。

正式 notebook 的最終分數 = cosine + bonus × [這頁是否被標題命中]。`True` 在 Python 裡等於 1、`False` 等於 0，所以 `bonus * is_title` 可以直接算。

In [ ]:
# 🧪 示範
pages = ['周杰倫', '專輯', '美國', '臺灣']
claim_vectors = F.normalize(torch.tensor([[1.0, 0.2], [0.1, 1.0]]), dim=1)
page_vectors = F.normalize(torch.tensor([[0.9, 0.3], [0.7, 0.7], [0.0, 1.0], [0.3, 0.9]]), dim=1)
similarities = claim_vectors @ page_vectors.T
print('相似度矩陣形狀：', tuple(similarities.shape))
print(similarities.round(decimals=3))
top = similarities.topk(k=2, dim=1)
for i, (values, indices) in enumerate(zip(top.values.tolist(), top.indices.tolist())):
    print(f'claim {i} 的前 2 名：', [(pages[j], round(v, 3)) for v, j in zip(values, indices)])

In [ ]:
# ✍️ 手打練習：rank_pages(candidates, bonus, k)
# candidates = {頁名: (cosine, 是否標題命中)}；依 cosine + bonus * 是否命中 由大到小排序，回傳前 k 個頁名
candidates = {'美國': (0.80, True), '周杰倫': (0.75, True), '專輯': (0.78, False), '臺灣': (0.30, False)}


def rank_pages(candidates, bonus, k):
    ranked = ...  # 提示：sorted(candidates.items(), key=lambda item: ..., reverse=True)
    return None   # 改成：取 ranked 前 k 個的頁名


check('bonus = 0', rank_pages(candidates, 0.0, 2), ['美國', '專輯'])
check('bonus = 0.1', rank_pages(candidates, 0.1, 2), ['美國', '周杰倫'])

<details><summary>參考答案</summary>

```python
def rank_pages(candidates, bonus, k):
    ranked = sorted(candidates.items(), key=lambda item: item[1][0] + bonus * item[1][1], reverse=True)
    return [page for page, _ in ranked[:k]]
```
`item` 是 `('美國', (0.80, True))`，所以 `item[1][0]` 是 cosine、`item[1][1]` 是是否命中。
bonus = 0 時「專輯」(0.78) 贏「周杰倫」(0.75)；bonus = 0.1 時周杰倫變 0.85，反超。這就是第 4 題消融實驗在比的東西。
</details>

## 5. 句子排序：local index → 全域 index ✍️

所有候選句放在一個大 list（`sentence_keys`）裡，每個 claim 只看「自己那 5 頁」的句子，先收集它們在大 list 的位置 `rows`。
對 `sentence_embeddings[rows]` 算分數、`topk` 之後，拿到的是 **rows 裡的位置（local）**，不是大 list 的位置。
一定要用 `rows[local]` 換回去，否則會拿到別頁的句子。這是整份程式最容易寫錯的地方。

In [ ]:
# 🧪 示範
sentence_keys = [('周杰倫', 0), ('周杰倫', 1), ('美國', 0), ('葉惠美', 0), ('葉惠美', 2)]
page_sentence_rows = {'周杰倫': [0, 1], '美國': [2], '葉惠美': [3, 4]}
rows = [row for page in ['葉惠美', '周杰倫'] for row in page_sentence_rows[page]]
print('這題的候選句在大 list 的位置 rows =', rows)          # [3, 4, 0, 1]
scores = torch.tensor([0.2, 0.9, 0.5, 0.1])                  # 依 rows 的順序算出來的分數
local = scores.topk(k=2).indices.tolist()
print('topk 回傳的是 local 位置：', local)                    # [1, 2]
print('❌ 直接拿去查：', [sentence_keys[i] for i in local])   # 錯！拿到 周杰倫 1、美國 0
print('✅ 先換回全域：', [sentence_keys[rows[i]] for i in local])

In [ ]:
# ✍️ 手打練習：top_sentences(pages, scores, k)
def top_sentences(pages, scores, k):
    rows = ...    # 把這些頁的句子位置串成一個 list
    best = ...    # scores 是依 rows 順序的分數 tensor，取前 k 名的 local 位置
    return None   # 換回全域位置後，回傳 [(頁名, 句號), ...]


check('top_sentences', top_sentences(['葉惠美', '周杰倫'], torch.tensor([0.2, 0.9, 0.5, 0.1]), 2),
      [('葉惠美', 2), ('周杰倫', 0)])

<details><summary>參考答案</summary>

```python
def top_sentences(pages, scores, k):
    rows = [row for page in pages for row in page_sentence_rows.get(page, [])]
    best = scores.topk(k=min(k, len(rows))).indices.tolist()
    return [sentence_keys[rows[local]] for local in best]
```
第一行是巢狀 list comprehension，等於 `for page in pages: for row in ...: rows.append(row)`。
</details>

## 6. bi-encoder vs cross-encoder（看懂就好）

| | bi-encoder（bge-m3） | cross-encoder（bge-reranker） |
| :--- | :--- | :--- |
| 怎麼算 | claim、句子**各自**變成向量，再算 cosine | `[CLS] claim [SEP] 句子 [SEP]` **一起**丟進模型，直接輸出一個分數 |
| 比喻 | 兩個人各自寫自我介紹，HR 只比對兩份履歷 | 兩個人坐下來面對面聊，HR 在旁邊聽 |
| 速度 | 快：句子向量算一次就能和所有 claim 比 | 慢：每一對 (claim, 句子) 都要重跑一次模型 |
| 準確度 | 粗：字詞沒有互相看到 | 細：claim 的每個字都能 attention 到句子的每個字 |
| 我的用法 | 先從 5 頁的所有句子挑 20 句 | 只重排這 20 句，取前 5 |

這種「先粗篩、再精排」是搜尋引擎的標準做法（retrieve → rerank）。

In [ ]:
# 🧪 示範：用「重疊的字數」假裝 cross-encoder，感受一下「一起看」和「各自看」的差別
claim = '周杰倫在2003年發行葉惠美'
candidates = ['周杰倫是台灣歌手', '葉惠美是周杰倫在2003年發行的專輯', '2003年發生了很多事']
for sentence in candidates:
    overlap = len(set(claim) & set(sentence))   # 真的 cross-encoder 會學到比字面重疊更聰明的東西
    print(f'{overlap:2d} 個字重疊  {sentence}')

## 7. 分類器的訓練資料：補進標準證據 ✍️

FEVER score 規定：SUPPORTS / REFUTES 的證據沒找到，標籤猜對也是 0 分。所以分類器要學的是
「**證據對的時候判斷對**」和「**證據不夠時選 NEI**」。訓練資料這樣給：

- S / R 題：檢索到的 5 句，如果沒包含一組標準證據，就把標準證據放進去、擠掉排名最後的句子，最後**打亂順序**。
- NEI 題：直接用檢索到的 5 句。

In [ ]:
# ✍️ 手打練習：inject_gold(retrieved, gold, k)
# retrieved：檢索到的 [(頁名, 句號), ...]（已排好名次）；gold：一組標準證據（sorted 過的 list）
# 回傳：gold + retrieved 裡不屬於 gold 的句子，總數補到 k 句（這裡先不打亂，方便對答案）
def inject_gold(retrieved, gold, k):
    others = ...  # retrieved 裡不在 gold 的
    return None   # gold + others 取到總共 k 句


retrieved = [('周杰倫', 0), ('美國', 0), ('周杰倫', 2), ('專輯', 1), ('臺灣', 0)]
check('標準證據不在檢索結果裡', inject_gold(retrieved, [('葉惠美', 1)], 5),
      [('葉惠美', 1), ('周杰倫', 0), ('美國', 0), ('周杰倫', 2), ('專輯', 1)])
check('標準證據已經在裡面', inject_gold(retrieved, [('周杰倫', 2)], 5),
      [('周杰倫', 2), ('周杰倫', 0), ('美國', 0), ('專輯', 1), ('臺灣', 0)])

<details><summary>參考答案</summary>

```python
def inject_gold(retrieved, gold, k):
    others = [key for key in retrieved if key not in gold]
    return gold + others[:max(0, k - len(gold))]
```
正式 notebook 之後還會 `rng.shuffle(keys)`，避免模型記住「標準證據永遠在第 1 個」。
</details>

## 8. 訓練迴圈 5 步驟 + fp16 + warmup ✍️

和 HW2 一樣的 5 步：`zero_grad → forward（算 loss）→ backward → clip → step`。這次多了兩個東西：

- **fp16 混合精度**：`torch.autocast` 讓前向用半精度（快、省記憶體）；`GradScaler` 先把 loss 放大再 backward，避免很小的梯度在 fp16 變成 0，更新前再縮回來（`unscale_`）。CPU 上它們會自動關掉，不影響理解。
- **warmup + 線性下降**：學習率前 10% 的步數從 0 慢慢升到 2e-5，之後線性降到 0。剛接上去的分類層是隨機的，一開始梯度很大，先用小步伐比較不會把預訓練好的權重弄壞。

下面用一個超小的模型在假資料上跑，看 loss 會不會下降。

In [ ]:
# 🧪 示範：學習率排程長什麼樣子
from transformers import get_linear_schedule_with_warmup

toy_model = torch.nn.Linear(4, 3)
toy_optimizer = torch.optim.AdamW(toy_model.parameters(), lr=2e-5)
toy_scheduler = get_linear_schedule_with_warmup(toy_optimizer, num_warmup_steps=2, num_training_steps=10)
learning_rates = []
for _ in range(10):
    learning_rates.append(toy_optimizer.param_groups[0]['lr'])
    toy_optimizer.step()
    toy_scheduler.step()
print('每一步的學習率：', [f'{lr:.1e}' for lr in learning_rates])

In [ ]:
# ✍️ 手打練習：補完 5 個步驟（不用 fp16，讓它在 CPU 上跑）
torch.manual_seed(0)
features, targets = torch.randn(64, 4), torch.randint(0, 3, (64,))
model = torch.nn.Linear(4, 3)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.05)
loss_fn = torch.nn.CrossEntropyLoss()
losses = []
for step in range(30):
    ...                                   # 1. 清梯度
    loss = loss_fn(model(features), targets)  # 2. 前向 + loss（已幫你寫好）
    ...                                   # 3. 反向傳播
    ...                                   # 4. 梯度裁剪（max_norm=1.0）
    ...                                   # 5. 更新參數
    losses.append(round(loss.item(), 3))
print('loss：', losses[0], '→', losses[-1])
print('✅ loss 有下降，5 個步驟都寫對了' if losses[-1] < losses[0] - 0.05 else '⏳ loss 沒有下降：還沒寫完或步驟有缺')

<details><summary>參考答案</summary>

```python
for step in range(30):
    optimizer.zero_grad()
    loss = loss_fn(model(features), targets)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
```
正式 notebook 用 fp16 時，第 3～5 步變成 `scaler.scale(loss).backward()` → `scaler.unscale_(optimizer)` → clip → `scaler.step(optimizer)` → `scaler.update()`，最後再 `scheduler.step()`。
</details>

## 9. Recall、FEVER score、NEI bias ✍️

| 指標 | 分母 | 怎樣算對 |
| :--- | :--- | :--- |
| Document Recall@5 | 只有 SUPPORTS / REFUTES | 前 5 頁包含某一組證據需要的**所有**頁 |
| Evidence Recall@5 | 只有 SUPPORTS / REFUTES | 前 5 句包含某一組證據的**所有**句（頁名 + 句號） |
| Accuracy | 全部 | 標籤對就好 |
| FEVER score | 全部 | 標籤對，而且（NEI 以外）前 5 句證據也要找齊 |

**NEI bias**：在 NEI 的 logit 加一個數字再取 argmax。加正數 → 更常猜 NEI。因為證據沒找到的 S/R 題本來就拿不到 FEVER 分數，
改猜 NEI 幾乎沒損失，卻可能多拿到真正 NEI 題的分數，所以 FEVER score 有可能上升、但 Accuracy 下降。

In [ ]:
# 🧪 示範：NEI bias 怎麼改變預測
LABELS = ['SUPPORTS', 'REFUTES', 'NOT ENOUGH INFO']
logits = torch.tensor([[2.0, 0.5, 1.6],
                       [0.1, 1.2, 1.0],
                       [0.3, 0.2, 0.1]])
for bias in [0.0, 0.3, 0.5]:
    adjusted = logits.clone()
    adjusted[:, 2] += bias
    print(f'bias = {bias}：', [LABELS[i] for i in adjusted.argmax(dim=1).tolist()])

In [ ]:
# ✍️ 手打練習：fever_score(rows)
# rows 每一筆 = (正確標籤, 標準證據組 [set, ...], 預測標籤, 預測的 5 句 [(頁名, 句號), ...])
def fever_score(rows):
    hits = 0
    for gold_label, groups, label, keys in rows:
        ...  # 標籤要對；不是 NEI 的話，前 5 句還要完整涵蓋任一組（提示：any(group <= set(keys[:5]) for group in groups)）
    return None  # 改成 hits / len(rows)


rows = [
    ('SUPPORTS', [{('周杰倫', 0)}], 'SUPPORTS', [('周杰倫', 0), ('美國', 1)]),            # 標籤對 + 證據對 → 1 分
    ('REFUTES', [{('李安', 2)}], 'REFUTES', [('李安', 0)]),                             # 標籤對但證據沒找到 → 0 分
    ('NOT ENOUGH INFO', [], 'NOT ENOUGH INFO', []),                                    # NEI 只看標籤 → 1 分
    ('SUPPORTS', [{('玉山', 0), ('臺灣', 3)}], 'SUPPORTS', [('玉山', 0), ('臺灣', 1)]),  # 一組要 2 句，只找到 1 句 → 0 分
]
check('fever_score', fever_score(rows), 0.5)

<details><summary>參考答案</summary>

```python
def fever_score(rows):
    hits = 0
    for gold_label, groups, label, keys in rows:
        if label == gold_label and (gold_label == 'NOT ENOUGH INFO'
                                    or any(group <= set(keys[:5]) for group in groups)):
            hits += 1
    return hits / len(rows)
```
第 2 筆是重點：Accuracy 會算它對，FEVER score 不會。這就是報告裡 Accuracy 和 FEVER score 數字不一樣的原因。
</details>

## 10. submission 的一列長怎樣（看懂就好）

`predicted_evidence` 要是 **JSON 字串**，不是 Python list。`ensure_ascii=False` 讓中文保持原樣，不然會變成 `周杰倫`。
NEI 一律填 `[]`。

In [ ]:
evidence = [('周杰倫', 0), ('葉惠美', 1)]
print('ensure_ascii=False：', json.dumps([[page, sid] for page, sid in evidence], ensure_ascii=False))
print('ensure_ascii=True ：', json.dumps([[page, sid] for page, sid in evidence]))
print('NEI：', json.dumps([]))